### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="african_credit_scoring",
    dataset_year="2024",
    domain_str="finance",
    # Data Source
    dataset_source="Zindi",
    original_dataset_source_download_link="https://zindi.africa/competitions/african-credit-scoring-challenge/data",
    download_description=r"""
We download the Kaggle upload of the Zindi challenge data.

kaggle datasets download thelastsmilodon/african-credit-scoring-challenge-data && unzip african-credit-scoring-challenge-data.zip && rm african-credit-scoring-challenge-data.zip
mkdir -p local-data-warehouse/african_credit_scoring && mv Train.csv Test.csv economic_indicators.csv VariableDefinitions.txt  local-data-warehouse/african_credit_scoring/
""",
    # References
    academic_reference_bibtex=r"""@misc{Zindi2024AfricanCreditScoringChallenge,
  author = {{Zindi}},
  title = {African Credit Scoring Challenge},
  year = {2024},
  howpublished = {\url{https://zindi.africa/competitions/african-credit-scoring-challenge}},
}
""",
    academic_reference_bibtex_key="Zindi2024AfricanCreditScoringChallenge",
    license="Other (specified in description)",
    data_tags=["Non-IID", "Grouped", "Temporal"],
    curation_comments="""
We curate this as a temporal credit risk task.

- The original challenge test split was partially random, partially grouped by country with samples from Ghana while test was Kenya only. 
    Since Train.csv only provides labeled Kenya rows, we therefore cannot reconstruct the official country-shift evaluation.
- The competition started on 2024-11-29, the last available disbursement date in the data is 2024-11-14. 
    Therefore, it looks like the data was pulled from the database shortly prior the competition using samples where it was already known that the loan was not paid at the due date, 
    without consideration of prediction time points and horizons. 
- The disbursement_date is very unevenly distributed, with most samples collected between July and November 2022, and only a few samples from earlier and later months. 
    Therefore, there might have been some preselction preventing proper time splits. Using just the latest 2024 samples is not accurate, 
    and splitting inside the high-density months also makes less sense since we would predict 2024 samples based on 2022 samples, 
    which is a very long horizon and also does not reflect the application where loans are issued continuously over time.
- A loan can stem from multiple lenders, where every lender is represented as one row in the dataframe. 
    This introduces leakage if splitting randomly. Also, loands stem from the same customers repeatedly using loans.
- tbl_loan_id is extremely correlated with the dates (when treated as numerical) (.99 spearman). Therefore, this feature is likely a proxy for credit issueing time.
- Whenever a customer has only default samples, there is also just one, or at most two samples for that customer. 
- Whenever a customer defaults, there is almost never another non-default in the data that happens later in time. 
    Therefore, a random split would introduce leakage and a grouped split would not reflect the application correctly.
- Hence, we define a temporal split by disbursement date because the weird date distribution is a less severe issue than the grouped leakage and temporal leakage that would be introduced by a random split.
    We separate the last 50% of disbursement dates into 9 buckets to be used as test data, and use all previous observations as train data for each split.
- We rename the target to "default".
- We convert dates to datetime.
- We drop ID and tbl_loan_id since they are categorical, but do not overlap with test.
- We drop country_id since it is always Kenya.
- To keep multi-loan information in the data we create a new feature "no_other_lenders_loan" indicating how many other lenders a customer has for a particular loan.

""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="default",
    problem_type="binary_classification",
    objective_metric_name="f1_score",
    stratify_on="default",
    time_on="disbursement_date",
    # group_on="customer_id",
    # group_labels="per_sample" # Might be per-group, need to check
)

## Preprocessing

In [2]:
import pandas as pd
df = pd.read_csv(dataset_mold.path / "Train.csv")
test = pd.read_csv(dataset_mold.path / "Test.csv")
economic = pd.read_csv(dataset_mold.path / "economic_indicators.csv")

# Convert dates to datetime
df["disbursement_date"] = pd.to_datetime(df["disbursement_date"])
df["due_date"] = pd.to_datetime(df["due_date"])
test["disbursement_date"] = pd.to_datetime(test["disbursement_date"])
test["due_date"] = pd.to_datetime(test["due_date"])


# df["ID_int"] = df["ID"].str.replace("ID_", "").astype(int)
# df["ID_int"] = df["ID_int"].apply(lambda x: int(str(x)[:-6])) # Remove lender_id integer part
# df["ID_int"] = df["ID_int"].apply(lambda x: int(str(x)[:-6])).astype(int) # Remove loan_id integer part

# df["num_disbursement_date"] = df["disbursement_date"].astype(int) // 10**9
# df["num_due_date"] = df["due_date"].astype(int) // 10**9

df = df.rename(columns={"target": "default"})

df["no_other_lenders_loan"] = df.groupby("tbl_loan_id")["ID"].transform("count")-1

df = df.drop(columns=["ID", "country_id"], errors="ignore")

cat_cols = ["customer_id", "lender_id", "loan_type", "New_versus_Repeat"]
for col in cat_cols:
    df[col] = df[col].astype("category")

print("Loaded data shape:", df.shape)

Loaded data shape: (68654, 15)


In [3]:
# cold_start_customers = df.loc[df["tbl_loan_id"].drop_duplicates().index, "customer_id"].value_counts()==1
# df["customer_id"].map(cold_start_customers).value_counts()

In [4]:
# # Whenever a customer defaults, there is almost never another non-default in the data that happens later in time. Therefore, a random split would introduce leakage.
# cnt = 0
# for customer in df["customer_id"].unique():
#     customer_rows = df[df["customer_id"] == customer]
#     if len(customer_rows) > 1 and customer_rows["default"].mean() not in [0, 1]:
#         print(f"{customer}: {customer_rows.sort_values('disbursement_date')['default'].values}")
#         cnt += 1

# import matplotlib.pyplot as plt
# # Whenever a customer defaults, there is almost never another non-default in the data that happens later in time. Therefore, a random split would introduce leakage.
# cnt = 0
# for customer in df["customer_id"].unique():
#     if cnt > 10:
#         break
#     customer_rows = df[df["customer_id"] == customer]
#     if len(customer_rows) > 1 and customer_rows["default"].mean() not in [0, 1]:
#         if customer_rows.sort_values("disbursement_date")["default"].values[-1] == 0:
#             plt.scatter(customer_rows["disbursement_date"], customer_rows["default"], marker="x")
#             plt.title(f"Customer {customer}")
#             plt.xlabel("Disbursement Date")
#             plt.ylabel("Default")
#             plt.show()

#             cnt += 1


In [5]:
# # Whenever a customer has only default samples, there is also just one, or at most two samples for that customer. 
# cnt = 0
# for customer in df["customer_id"].unique():
#     customer_rows = df[df["customer_id"] == customer]
#     if len(customer_rows) > 1 and customer_rows["target"].mean() == 1:
#         print(f"{customer}: {customer_rows.sort_values('disbursement_date')['target'].values}")
#         cnt += 1


In [6]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

,customer_id,tbl_loan_id,lender_id,loan_type,Total_Amount,Total_Amount_to_Repay,disbursement_date,due_date,duration,New_versus_Repeat,Amount_Funded_By_Lender,Lender_portion_Funded,Lender_portion_to_be_repaid,default,no_other_lenders_loan
0,266671,248032,267278,Type_1,8448.0,8448.0,2022-08-30,2022-09-06,7,Repeat Loan,120.85,0.014305,121.0,0,0
1,248919,228515,267278,Type_1,25895.0,25979.0,2022-07-30,2022-08-06,7,Repeat Loan,7768.50,0.300000,7794.0,0,0
2,308486,370501,251804,Type_7,6900.0,7142.0,2024-09-06,2024-09-13,7,Repeat Loan,1380.00,0.200000,1428.0,0,0
3,266004,285009,267278,Type_1,8958.0,9233.0,2022-10-20,2022-10-27,7,Repeat Loan,2687.40,0.300000,2770.0,0,0
4,253803,305312,267278,Type_1,4564.0,4728.0,2022-11-28,2022-12-05,7,Repeat Loan,1369.20,0.300000,1418.0,0,0


## Data Checks

In [7]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 68,654
Columns: 15
Use sampling: False (sample size: 68,654)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['tbl_loan_id', 'Amount_Funded_By_Lender', 'Total_Amount_to_Repay', 'Total_Amount', 'Lender_portion_Funded', 'Lender_portion_to_be_repaid', 'customer_id', 'due_date', 'disbursement_date', 'duration']
Rows remaining as candidates after top-10 filter: 3,726 (of 68,654)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [8]:
# Sample Rows
df_head

,customer_id,tbl_loan_id,lender_id,loan_type,Total_Amount,Total_Amount_to_Repay,disbursement_date,due_date,duration,New_versus_Repeat,Amount_Funded_By_Lender,Lender_portion_Funded,Lender_portion_to_be_repaid,default,no_other_lenders_loan
0,266671,248032,267278,Type_1,8448.0,8448.0,2022-08-30,2022-09-06,7,Repeat Loan,120.85,0.014305,121.0,0,0
1,248919,228515,267278,Type_1,25895.0,25979.0,2022-07-30,2022-08-06,7,Repeat Loan,7768.50,0.300000,7794.0,0,0
2,308486,370501,251804,Type_7,6900.0,7142.0,2024-09-06,2024-09-13,7,Repeat Loan,1380.00,0.200000,1428.0,0,0
3,266004,285009,267278,Type_1,8958.0,9233.0,2022-10-20,2022-10-27,7,Repeat Loan,2687.40,0.300000,2770.0,0,0
4,253803,305312,267278,Type_1,4564.0,4728.0,2022-11-28,2022-12-05,7,Repeat Loan,1369.20,0.300000,1418.0,0,0


In [9]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,customer_id,category,0.0,0.0,6540.0,"247613, 250874, 259757, 255356, 249457, 262000, 260083, 253737, 254014, 246888"
1,lender_id,category,0.0,0.0,4.0,"267278, 251804, 267277, 245684"
2,loan_type,category,0.0,0.0,22.0,"Type_1, Type_7, Type_5, Type_4, Type_10, Type_6, Type_9, Type_14, Type_2, Type_11"
3,New_versus_Repeat,category,0.0,0.0,2.0,"Repeat Loan, New Loan"
4,disbursement_date,datetime64[ns],0.0,0.0,768.0,"2022-07-16 00:00:00, 2022-07-25 00:00:00, 2022-08-01 00:00:00, 2022-07-23 00:00:00, 2022-07-30 00:00:00, 2022-09-19 00:00:00, 2022-07-18 00:00:00, 2022-07-27 00:00:00, 2022-10-03 00:00:00, 2022-07-13 00:00:00"
5,due_date,datetime64[ns],0.0,0.0,893.0,"2022-07-23 00:00:00, 2022-08-01 00:00:00, 2022-08-08 00:00:00, 2022-07-30 00:00:00, 2022-08-06 00:00:00, 2022-09-26 00:00:00, 2022-07-25 00:00:00, 2022-08-03 00:00:00, 2022-07-20 00:00:00, 2022-08-09 00:00:00"
6,Total_Amount,float64,0.0,0.0,19076.0,"1500.0, 5000.0, 10000.0, 4699.0, 2199.0, 2000.0, 2250.0, 4989.0, 6000.0, 6499.0"
7,Total_Amount_to_Repay,float64,0.0,0.0,21920.0,"5176.0, 1555.0, 1500.0, 2199.0, 4699.0, 10700.0, 2250.0, 6211.0, 6499.0, 4989.0"
8,Amount_Funded_By_Lender,float64,0.0,0.0,23391.0,"0.0, 450.0, 1000.0, 1200.0, 600.0, 659.7, 1496.7, 1949.7, 1409.7, 10000.0"
9,Lender_portion_Funded,float64,0.0,0.0,12844.0,"0.3, 0.0, 0.2, 0.16, 0.5, 0.1333, 0.25, 1.0, 0.2375, 0.1333"


In [10]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
tbl_loan_id,68654.0,263056.266248,39486.661487,101323.0,3.753200e+05
Total_Amount,68654.0,14836.828617,141649.868388,2.0,2.300000e+07
Total_Amount_to_Repay,68654.0,15639.929901,165078.352830,0.0,2.541500e+07
duration,68654.0,8.544586,13.343145,1.0,1.096000e+03
Amount_Funded_By_Lender,68654.0,2545.663204,11922.724169,0.0,1.600000e+06
Lender_portion_Funded,68654.0,0.218679,0.129832,0.0,1.168119e+00
Lender_portion_to_be_repaid,68654.0,2652.621493,13380.063537,0.0,1.821338e+06
default,68654.0,0.018324,0.134120,0.0,1.000000e+00
no_other_lenders_loan,68654.0,0.062167,0.241460,0.0,1.000000e+00


In [11]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column            rank                                   
New_versus_Repeat 1             Repeat Loan  68087  99.17
                  2                New Loan    567   0.83
customer_id       1                  247613    208   0.30
                  2                  250874    192   0.28
                  3                  259757    135   0.20
                  4                  255356    128   0.19
                  5                  249457    119   0.17
disbursement_date 1     2022-07-16 00:00:00    938   1.37
                  2     2022-07-25 00:00:00    887   1.29
                  3     2022-08-01 00:00:00    884   1.29
                  4     2022-07-23 00:00:00    872   1.27
                  5     2022-07-30 00:00:00    853   1.24
due_date          1     2022-07-23 00:00:00    940   1.37
                  2     2022-08-01 00:00:00    887   1.29
                  3     2022-08-08 00:00:00    885   1.29
                  4     2022-07-30 00:00:00    871   1.27
                  5     2022-08-06 00:00:00    865   1.26
lender_id         1                  267278  64653  94.17
                  2                  251804   3542   5.16
                  3                  267277    271   0.39
                  4                  245684    188   0.27
loan_type         1                  Type_1  61723  89.90
                  2                  Type_7   2790   4.06
                  3                  Type_5   1521   2.22
                  4                  Type_4   1235   1.80
                  5                 Type_10    466   0.68

In [12]:
# Target Distribution
target_df

,count,pct
default,,
0,67396,98.17
1,1258,1.83


## Task Curation

In [13]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

ValueError: We cannot provide recommend split dimensions for time-based splits. Judge the appropriate time horizon manually!

In [ ]:
import numpy as np
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

all_test_splits = df[df.disbursement_date>df.disbursement_date.median()]

s = all_test_splits["disbursement_date"].sort_values()

d = s.dt.normalize()

counts = d.value_counts().sort_index()
cum = counts.cumsum()

targets = np.linspace(0, len(s), 10)[1:-1]

cut_dates = []
last = pd.Timestamp.min

for t in targets:
    candidate = cum[cum >= t].index[0]
    if candidate > last:
        cut_dates.append(candidate)
        last = candidate

bins = [pd.Timestamp.min] + cut_dates + [pd.Timestamp.max]

test_idx = [
    s[(d > bins[i]) & (d <= bins[i + 1])].index.tolist()
    for i in range(len(bins) - 1)
]

used_in_train = set()
used_in_test = set()
used_data = set()
splits = {}

target_col = task_mold.target_column_name

for split, idx in enumerate(test_idx[::-1]):
    test_df = df.loc[idx]
    train_df = df.loc[df.disbursement_date < test_df["disbursement_date"].min() - pd.DateOffset(days=3)]

    train_idx = train_df.index.tolist()
    test_idx = test_df.index.tolist()

    splits[split] = {0: [train_idx, test_idx]}

    used_in_train.update(train_idx)
    used_in_test.update(test_idx)
    used_data.update(train_idx)
    used_data.update(test_idx)

    print(f"\n=== Step {split} ===")
    print("Train size:", len(train_idx), "| Test size:", len(test_idx))
    print("Train target mean:", df.loc[train_idx, target_col].astype(int).mean())
    print("Test target mean:", df.loc[test_idx, target_col].astype(int).mean())

    assert len(set(train_idx).intersection(set(test_idx)))==0, "Train and test indices overlap!"
    assert len(set(df.loc[train_idx].tbl_loan_id.unique()).intersection(set(df.loc[test_idx].tbl_loan_id.unique()))) == 0, "Loan leakage between train and test!"   

print(f"{len(used_data)/df.shape[0]:.4f} of the samples are used.")
print(f"{len(used_in_train)/df.shape[0]:.4f} of the samples are used in training")
print(f"{len(used_in_test)/df.shape[0]:.4f} of the samples are used in testing.")

df = df.drop(columns=["tbl_loan_id"], errors="ignore")

splits_mold = PredictiveMLSplitsMetadata( 
    splits_comment=r"To define train/test splits, we separate the last 50% of disbursement dates into 9 buckets to be used as test data, and use all previous observations as train data for each split.",
    splits=splits,
)


=== Step 0 ===
Train size: 64878 | Test size: 3754
Train target mean: 0.013995499244736274
Test target mean: 0.09216835375599361

=== Step 1 ===
Train size: 60397 | Test size: 3739
Train target mean: 0.012997334304684008
Test target mean: 0.03236159400909334

=== Step 2 ===
Train size: 56514 | Test size: 3568
Train target mean: 0.013217963690412995
Test target mean: 0.010089686098654708

=== Step 3 ===
Train size: 52451 | Test size: 3940
Train target mean: 0.013402985643743684
Test target mean: 0.009898477157360405

=== Step 4 ===
Train size: 48640 | Test size: 3628
Train target mean: 0.01342516447368421
Test target mean: 0.011025358324145534

=== Step 5 ===
Train size: 45058 | Test size: 3738
Train target mean: 0.013893204314439167
Test target mean: 0.011235955056179775

=== Step 6 ===
Train size: 41657 | Test size: 3291
Train target mean: 0.014451352713829609
Test target mean: 0.005165603160133698

=== Step 7 ===
Train size: 37146 | Test size: 4269
Train target mean: 0.0147795186561

In [ ]:
'''Alternative split idea that was rejected later on due to introducing too strong target shifts.
- We need a split that 
    a) avoids temporal leakage such that no non-default samples appear after a default for customers, 
    b) avoids grouped leakage for tbl_loan_id, 
    c) accounts for the weird distribution of disbursement dates preventing a proper temporal split, 
    d) Reflects the application where loans are issued over time and the prediction point is always before a loan is issued and previous information about a customer can be available.
- Based on this considerations, we define a chronological leave-last-loan-out per customer split to focus the task on individual customer histories.
    1. We split the data into train/test by customer. 
    2. For each customer, we sort loans by disbursement_date and use the last observed tbl_loan_id as test sample, marking the candidate test loan.
    3. We keep all earlier loans of the same customer in train, to reflect the application where previous information about a customer can be available at decision time.
'''

# from data_foundry.schema import PredictiveMLSplitsMetadata
# from data_foundry import curation_recommendations

# splits = curation_recommendations.get_recommended_grouped_splits(
#     dataset=df,
#     n_repeats=n_repeats,
#     n_splits=n_splits,
#     group_on=task_mold.group_on,
#     test_size=none_or_test_size,
#     stratify_on=task_mold.stratify_on,
#     group_labels=task_mold.group_labels,
#     show_splits=True,
#     target_on=task_mold.target_column_name,
# )

# last_test_date_per_customer = df.groupby("customer_id")["disbursement_date"].max()

# group_on = "customer_id"
# target_on = task_mold.target_column_name

# for repeat_idx in range(n_repeats):
#     for fold_idx in range(n_splits):
#         train_idx, test_idx = splits[repeat_idx][fold_idx]
#         df_train = df.loc[train_idx]
#         df_test = df.loc[test_idx]  
    
#         assert len(set(df.loc[train_idx].customer_id.unique()).intersection(set(df.loc[test_idx].customer_id.unique()))) == 0, "Customer leakage between train and test!"
#         assert len(set(df.loc[train_idx].tbl_loan_id.unique()).intersection(set(df.loc[test_idx].tbl_loan_id.unique()))) == 0, "Loan leakage between train and test!"   

#         test_idx = df_test.index[df_test["disbursement_date"]>(df_test.customer_id.map(last_test_date_per_customer)-pd.Timedelta(days=7))].tolist()
#         train_idx = df_test.index.difference(test_idx).tolist() + train_idx

#         train_target_dist = df.iloc[train_idx][target_on].mean()
#         test_target_dist = df.iloc[test_idx][target_on].mean()

#         print(f"""Repeat {repeat_idx}, Fold {fold_idx}:
#         Train N: {len(train_idx)}, Test N: {len(test_idx)}
#         Target Distribution:
#         \tTrain target distribution: {train_target_dist}
#         \tTest target distribution: {test_target_dist}
#         Group Distribution {group_on}:
#         \tTrain: {len(df.iloc[train_idx][group_on].unique())}
#         \tTest: {len(df.iloc[test_idx][group_on].unique())}
#         """)



# # df["disbursement_date"].apply(lambda x: 
# last_test_date_per_customer

'Alternative split idea that was rejected later on due to introducing too strong target shifts.\n- We need a split that \n    a) avoids temporal leakage such that no non-default samples appear after a default for customers, \n    b) avoids grouped leakage for tbl_loan_id, \n    c) accounts for the weird distribution of disbursement dates preventing a proper temporal split, \n    d) Reflects the application where loans are issued over time and the prediction point is always before a loan is issued and previous information about a customer can be available.\n- Based on this considerations, we define a chronological leave-last-loan-out per customer split to focus the task on individual customer histories.\n    1. We split the data into train/test by customer. \n    2. For each customer, we sort loans by disbursement_date and use the last observed tbl_loan_id as test sample, marking the candidate test loan.\n    3. We keep all earlier loans of the same customer in train, to reflect the app

## Export

In [ ]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to african_credit_scoring_challenge_data/019dc44f-8afd-72c2-bcee-bb21c5902173
019dc44f-8afd-72c2-bcee-bb21c5902173
6995ebf5a9b01b864d3cb72418c1d824f243439ecc1676b6d3b5a042f0ff318d
